In [2]:
from google.colab import drive

drive.mount('/content/drive')
path = '/content/drive/MyDrive/Datasheet/movies_metadata.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd

df = pd.read_csv(path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

/tmp/ipykernel_25568/3766738840.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


In [4]:
feature = ['title','genres','overview','tagline','belongs_to_collection']
df_feature = df[feature].copy()

In [5]:
df_feature.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   title                  45460 non-null  object
 1   genres                 45466 non-null  object
 2   overview               44512 non-null  object
 3   tagline                20412 non-null  object
 4   belongs_to_collection  4494 non-null   object
dtypes: object(5)
memory usage: 1.7+ MB


**Data Cleaning**

In [6]:
df_feature.head()

,title,genres,overview,tagline,belongs_to_collection
0,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","Led by Woody, Andy's toys live happily in his ...",NaN,"{'id': 10194, 'name': 'Toy Story Collection', ..."
1,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",When siblings Judy and Peter discover an encha...,Roll the dice and unleash the excitement!,NaN
2,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,"{'id': 119050, 'name': 'Grumpy Old Men Collect..."
3,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,NaN
4,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",Just when George Banks has recovered from his ...,Just When His World Is Back To Normal... He's ...,"{'id': 96871, 'name': 'Father of the Bride Col..."


In [7]:
df_feature['genres'].apply(type).value_counts(dropna=False)

,count
genres,
<class 'str'>,45466


In [8]:
df_feature['belongs_to_collection'].apply(type).value_counts(dropna=False)

,count
belongs_to_collection,
<class 'float'>,40972
<class 'str'>,4494


In [9]:
import ast

def get_extraction_name(x):
  if isinstance(x, float):
    return ""

  # Proses ektrasi name "kunci"
  try:
    data_collection = ast.literal_eval(x)
    return data_collection['name']
  except (ValueError, SyntaxError, KeyError, TypeError):
    return ""

In [10]:
df_feature['genres'] = df_feature['genres'].apply(lambda x: " ".join([i['name']for i in ast.literal_eval(x)]))

In [11]:
df_feature['belongs_to_collection'] = df_feature['belongs_to_collection'].apply(get_extraction_name)

In [12]:
df_feature.head()

,title,genres,overview,tagline,belongs_to_collection
0,Toy Story,Animation Comedy Family,"Led by Woody, Andy's toys live happily in his ...",NaN,Toy Story Collection
1,Jumanji,Adventure Fantasy Family,When siblings Judy and Peter discover an encha...,Roll the dice and unleash the excitement!,
2,Grumpier Old Men,Romance Comedy,A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,Grumpy Old Men Collection
3,Waiting to Exhale,Comedy Drama Romance,"Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,
4,Father of the Bride Part II,Comedy,Just when George Banks has recovered from his ...,Just When His World Is Back To Normal... He's ...,Father of the Bride Collection


In [13]:
df_feature.duplicated(subset=['title','genres','overview','tagline'], keep='first').sum()

np.int64(32)

In [14]:
dup_suspect = df_feature[df_feature.duplicated(subset=['title','genres','overview','tagline'], keep=False)].sort_values(by=['title'])

In [15]:
pd.set_option('display.max_rows',None)
display(dup_suspect[['title','genres','overview','tagline','belongs_to_collection']])

,title,genres,overview,tagline,belongs_to_collection
32791,12 Chairs,Adventure Comedy,A former aristocrat Ippolit Vorobyaninov leads...,NaN,
32777,12 Chairs,Adventure Comedy,A former aristocrat Ippolit Vorobyaninov leads...,NaN,
15074,A Farewell to Arms,Drama Romance War,British nurse Catherine Barkley (Helen Hayes) ...,Every woman who has loved will understand,
949,A Farewell to Arms,Drama Romance War,British nurse Catherine Barkley (Helen Hayes) ...,Every woman who has loved will understand,
21116,A Place at the Table,Documentary,"Using personal stories, this powerful document...",One Nation. Underfed.,
2564,A Place at the Table,Documentary,"Using personal stories, this powerful document...",One Nation. Underfed.,
22733,A Streetcar Named Desire,,Blanche Dubois goes to visit her pregnant sist...,NaN,
22734,A Streetcar Named Desire,,Blanche Dubois goes to visit her pregnant sist...,NaN,
11155,Black Gold,Adventure Drama,"On the Arabian Peninsula in the 1930s, two war...",NaN,
20843,Black Gold,Adventure Drama,"On the Arabian Peninsula in the 1930s, two war...",NaN,


In [16]:
df_feature = df_feature.drop_duplicates(subset=['title','genres','overview'])

In [17]:
df_feature.duplicated(subset=['title','genres','overview','tagline']).sum()

np.int64(0)

In [18]:
df_feature.duplicated().sum()

np.int64(0)

In [19]:
display(dup_suspect[['title','genres','overview','tagline','belongs_to_collection']])

,title,genres,overview,tagline,belongs_to_collection
32791,12 Chairs,Adventure Comedy,A former aristocrat Ippolit Vorobyaninov leads...,NaN,
32777,12 Chairs,Adventure Comedy,A former aristocrat Ippolit Vorobyaninov leads...,NaN,
15074,A Farewell to Arms,Drama Romance War,British nurse Catherine Barkley (Helen Hayes) ...,Every woman who has loved will understand,
949,A Farewell to Arms,Drama Romance War,British nurse Catherine Barkley (Helen Hayes) ...,Every woman who has loved will understand,
21116,A Place at the Table,Documentary,"Using personal stories, this powerful document...",One Nation. Underfed.,
2564,A Place at the Table,Documentary,"Using personal stories, this powerful document...",One Nation. Underfed.,
22733,A Streetcar Named Desire,,Blanche Dubois goes to visit her pregnant sist...,NaN,
22734,A Streetcar Named Desire,,Blanche Dubois goes to visit her pregnant sist...,NaN,
11155,Black Gold,Adventure Drama,"On the Arabian Peninsula in the 1930s, two war...",NaN,
20843,Black Gold,Adventure Drama,"On the Arabian Peninsula in the 1930s, two war...",NaN,


In [20]:
df_feature.isna().sum()

,0
title,6
genres,0
overview,954
tagline,25033
belongs_to_collection,0


In [21]:
missing_overview_tagline = df_feature[df_feature['overview'].isna() | df_feature['tagline'].isna()]
display(missing_overview_tagline[['title','genres','overview','tagline']].head(5))

,title,genres,overview,tagline
0,Toy Story,Animation Comedy Family,"Led by Woody, Andy's toys live happily in his ...",NaN
11,Dracula: Dead and Loving It,Comedy Horror,When a lawyer shows up at the vampire's doorst...,NaN
27,Persuasion,Drama Romance,This film adaptation of Jane Austen's last nov...,NaN
32,Wings of Courage,Romance Adventure,NaN,NaN
35,Dead Man Walking,Drama,A justice drama based on a true story about a ...,NaN


In [22]:
df_feature.loc[:, 'overview'] = df_feature['overview'].fillna("")
df_feature.loc[:, 'tagline'] = df_feature['tagline'].fillna("")

In [23]:
df_feature.isna().sum()

,0
title,6
genres,0
overview,0
tagline,0
belongs_to_collection,0


In [24]:
missing_title = df_feature[df_feature['title'].isna()]
display(missing_title)

,title,genres,overview,tagline,belongs_to_collection
19729,NaN,Action Thriller Drama,British soldiers force a recently captured IRA...,,
19730,NaN,Carousel Productions Vision View Entertainment...,Released,,
29502,NaN,Animation Science Fiction,Third film of the Mardock Scramble series.,,Mardock Scramble Collection
29503,NaN,Aniplex GoHands BROSTA TV Mardock Scramble Pro...,Released,,
35586,NaN,TV Movie Action Horror Science Fiction,A group of skiers are terrorized during spring...,,
35587,NaN,Odyssey Media Pulser Productions Rogue State T...,Released,,


In [25]:
df_feature = df_feature.dropna(subset=['title'])

In [26]:
df_feature.isna().sum()

,0
title,0
genres,0
overview,0
tagline,0
belongs_to_collection,0


In [27]:
df_feature.shape

(45428, 5)

In [28]:
df_feature.head()

,title,genres,overview,tagline,belongs_to_collection
0,Toy Story,Animation Comedy Family,"Led by Woody, Andy's toys live happily in his ...",,Toy Story Collection
1,Jumanji,Adventure Fantasy Family,When siblings Judy and Peter discover an encha...,Roll the dice and unleash the excitement!,
2,Grumpier Old Men,Romance Comedy,A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,Grumpy Old Men Collection
3,Waiting to Exhale,Comedy Drama Romance,"Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,
4,Father of the Bride Part II,Comedy,Just when George Banks has recovered from his ...,Just When His World Is Back To Normal... He's ...,Father of the Bride Collection


**Text Preprocessing**

In [29]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

text_features = [
    'genres',
    'overview',
    'tagline',
    'belongs_to_collection'
]
content_df = df_feature[
    ['title'] + text_features
].copy()

In [30]:
# Hapus spasi berlebihan
for col in text_features:
  content_df[col] = content_df[col].str.strip()
content_df.head(10)

,title,genres,overview,tagline,belongs_to_collection
0,Toy Story,Animation Comedy Family,"Led by Woody, Andy's toys live happily in his ...",,Toy Story Collection
1,Jumanji,Adventure Fantasy Family,When siblings Judy and Peter discover an encha...,Roll the dice and unleash the excitement!,
2,Grumpier Old Men,Romance Comedy,A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,Grumpy Old Men Collection
3,Waiting to Exhale,Comedy Drama Romance,"Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,
4,Father of the Bride Part II,Comedy,Just when George Banks has recovered from his ...,Just When His World Is Back To Normal... He's ...,Father of the Bride Collection
5,Heat,Action Crime Drama Thriller,"Obsessive master thief, Neil McCauley leads a ...",A Los Angeles Crime Saga,
6,Sabrina,Comedy Romance,An ugly duckling having undergone a remarkable...,You are cordially invited to the most surprisi...,
7,Tom and Huck,Action Adventure Drama Family,"A mischievous young boy, Tom Sawyer, witnesses...",The Original Bad Boys.,
8,Sudden Death,Action Adventure Thriller,International action superstar Jean Claude Van...,Terror goes into overtime.,
9,GoldenEye,Adventure Action Thriller,James Bond must unmask the mysterious head of ...,No limits. No fears. No substitutes.,James Bond Collection


In [32]:
tfidf_matrices = {}
for col in text_features:
  vectorizer = TfidfVectorizer(
      stop_words="english",
      max_features=5000,
      dtype=np.float32
  )

  tfidf_matrices[col] = vectorizer.fit_transform(content_df[col])
  print(
      f"{col} : {tfidf_matrices[col].shape}, "
      f"non-zero values: {tfidf_matrices[col].nnz:,}"
  )

genres : (45428, 22), non-zero values: 94,811
overview : (45428, 5000), non-zero values: 914,184
tagline : (45428, 5000), non-zero values: 75,286
belongs_to_collection : (45428, 2276), non-zero values: 12,372


In [68]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def content_based_recommender(
    movie_title, data, tfidf_matrices, top_n=10
):
    weights = {
        "genres": 0.30,
        "overview": 0.45,
        "tagline": 0.10,
        "belongs_to_collection": 0.15
    }

    # Cari posisi baris film, bukan label indeks DataFrame
    titles = data["title"].fillna("").str.casefold().to_numpy()
    matches = np.flatnonzero(titles == movie_title.casefold())

    if len(matches) == 0:
        print(f"Film '{movie_title}' tidak ditemukan.")
        return data.iloc[0:0].copy()

    movie_idx = int(matches[0])
    scores = np.zeros(len(data), dtype=np.float32)

    for col, weight in weights.items():
        matrix = tfidf_matrices[col]

        if matrix.shape[0] != len(data):
            raise ValueError(
                f"Jumlah baris matriks {col} tidak sama "
                "dengan jumlah baris data."
            )

        similarity = cosine_similarity(
            matrix[movie_idx], matrix
        ).ravel()

        scores += weight * similarity.astype(np.float32)

    # Keluarkan film input dari hasil rekomendasi
    scores[movie_idx] = -1

    top_n = min(top_n, len(data) - 1)
    top_indices = np.argsort(scores)[-top_n:][::-1]

    results = data.iloc[top_indices].copy()
    results["similarity_score"] = scores[top_indices]

    return results.reset_index(drop=True)

In [64]:
recommendations = content_based_recommender(
    movie_title="Toy Story",
    data=content_df,
    tfidf_matrices=tfidf_matrices,
    top_n=10
)

In [65]:
recommendations[
    [
        'title',
        'genres',
        'belongs_to_collection',
        'similarity_score'
    ]
]

,title,genres,belongs_to_collection,similarity_score
0,Toy Story 3,Animation Family Comedy,Toy Story Collection,0.665822
1,Toy Story 2,Animation Comedy Family,Toy Story Collection,0.597536
2,Andy Hardy's Blonde Trouble,Comedy Family Romance,,0.362152
3,Hawaiian Vacation,Animation Family,,0.355225
4,"Monsters, Inc.",Animation Comedy Family,"Monsters, Inc. Collection",0.350465
5,Free Birds,Animation Comedy Family,,0.344598
6,Garfield: A Tail of Two Kitties,Animation Comedy Family,Garfield Collection,0.328715
7,The Wild,Family Animation,,0.328301
8,Monster High: Welcome to Monster High,Animation Family,Monster High Collection,0.327672
9,"Rudolph, the Red-Nosed Reindeer",Animation Family,Rudolph the Red-Nosed Reindeer Collection,0.325905


In [47]:
recommendations = content_based_recommender(
    movie_title="The Shawshank Redemption",
    data=content_df,
    tfidf_matrices=tfidf_matrices,
    top_n=10
)

display(
    recommendations[
        ["title", "genres", "belongs_to_collection", "similarity_score"]
    ]
)

,title,genres,belongs_to_collection,similarity_score
0,Brute Force,Crime Drama,,0.418596
1,Breath,Crime Drama,,0.398099
2,Duffy of San Quentin,Drama Crime,,0.393072
3,Brubaker,Crime Drama,,0.387561
4,Variety,Drama Crime,,0.386127
5,Cool Hand Luke,Crime Drama,,0.383035
6,Convict's Code,Crime Drama,,0.381893
7,A Prophet,Crime Drama,,0.381181
8,The Domino Principle,Drama Crime,,0.379429
9,House of Women,Crime Drama,,0.374816


In [66]:
movie_title = "The Shawshank Redemption"

# Tampilkan sinopsis film input
display(
    content_df.loc[
        content_df["title"].str.casefold() == movie_title.casefold(),
        ["title", "genres", "overview"]
    ]
)

,title,genres,overview
314,The Shawshank Redemption,Drama Crime,Framed in the 1940s for the double murder of h...


In [52]:
# Tampilkan sinopsis hasil rekomendasi
display(
    recommendations[
        ["title", "genres", "overview", "similarity_score"]
    ]
)

,title,genres,overview,similarity_score
0,Brute Force,Crime Drama,Prison inmates revolt against a sadistic guard.,0.418596
1,Breath,Crime Drama,A convicted prisoner falls for a woman that de...,0.398099
2,Duffy of San Quentin,Drama Crime,San Quentin's new warden crusades for reform a...,0.393072
3,Brubaker,Crime Drama,The new warden of a small prison farm in Arkan...,0.387561
4,Variety,Drama Crime,The murderer “Boss” Huller – after having spen...,0.386127
5,Cool Hand Luke,Crime Drama,When petty criminal Luke Jackson is sentenced ...,0.383035
6,Convict's Code,Crime Drama,"On parole after three years in prison, a footb...",0.381893
7,A Prophet,Crime Drama,"Sentenced to six years in prison, Malik El Dje...",0.381181
8,The Domino Principle,Drama Crime,Tucker is a chronic underachiever and a loser....,0.379429
9,House of Women,Crime Drama,"An innocent, pregnant prison inmate (Shirley K...",0.374816


In [67]:
display(
    (content_based_recommender(
        'The Dark Knight',
        content_df,
        tfidf_matrices,
        top_n=10
    )[
        ['title','genres','similarity_score']
    ])
)

,title,genres,similarity_score
0,The Dark Knight Rises,Action Crime Drama Thriller,0.576426
1,Batman Begins,Action Crime Drama,0.489785
2,Ricochet,Action Crime Thriller,0.361820
3,Mesrine: Killer Instinct,Drama Action Thriller Crime,0.355774
4,Get the Gringo,Action Thriller Crime Drama,0.344987
5,Running Out of Time,Drama Action Thriller Crime,0.344874
6,Hitman,Action Crime Drama Thriller,0.341553
7,S.W.A.T.,Action Thriller Crime,0.341496
8,Mesrine: Public Enemy #1,Action Thriller Crime Drama,0.338834
9,F/X,Crime Thriller Action,0.336366


In [69]:
test_movies = [
    'The Dark Knight',
    'Finding Nemo',
    'The GodFather',
    'Titanic',
    'Toy Story',
    'The Conjuring',
    'Home Alone',
    'The Matrix',
    'Minions',
    'The Shawshank Redemption'
]

In [70]:
for movies in test_movies:
  print(f"\n{'=' * 80}")
  print(f"REKOMENDASI UNTUK : {movies}")
  print(f"{'=' * 80}")
  result = content_based_recommender(
      movie_title=movies,
      data=content_df,
      tfidf_matrices=tfidf_matrices,
      top_n = 10
  )

  display(result[['title','genres','belongs_to_collection','similarity_score']])


REKOMENDASI UNTUK : The Dark Knight


,title,genres,belongs_to_collection,similarity_score
0,The Dark Knight Rises,Action Crime Drama Thriller,The Dark Knight Collection,0.576426
1,Batman Begins,Action Crime Drama,The Dark Knight Collection,0.489785
2,Ricochet,Action Crime Thriller,,0.361820
3,Mesrine: Killer Instinct,Drama Action Thriller Crime,The Mesrine Collection,0.355774
4,Get the Gringo,Action Thriller Crime Drama,,0.344987
5,Running Out of Time,Drama Action Thriller Crime,Running Out Of Time Collection,0.344874
6,Hitman,Action Crime Drama Thriller,Hitman Collection,0.341553
7,S.W.A.T.,Action Thriller Crime,S.W.A.T. Collection,0.341496
8,Mesrine: Public Enemy #1,Action Thriller Crime Drama,The Mesrine Collection,0.338834
9,F/X,Crime Thriller Action,F/X Collection,0.336366



REKOMENDASI UNTUK : Finding Nemo


,title,genres,belongs_to_collection,similarity_score
0,Back To The Sea,Animation Family,,0.402791
1,Finding Dory,Adventure Animation Comedy Family,Finding Nemo Collection,0.390276
2,The Reef 2: High Tide,Animation Family,The Reef Collection,0.386646
3,The Secret of NIMH 2: Timmy to the Rescue,Animation Family,The Secret Of NIMH Collection,0.350563
4,Ozzy,Animation Family,,0.350451
5,The Rugrats Movie,Animation Family,Rugrats Collection,0.348699
6,Paperman,Animation Family Romance,,0.345272
7,VeggieTales: Where's God When I'm Scared,Animation Family,,0.343677
8,The Littlest Angel,Animation Family,,0.343335
9,Bambi II,Animation Drama Family,Bambi Collection,0.340213



REKOMENDASI UNTUK : The GodFather


...

,title,genres,belongs_to_collection,similarity_score
0,The Matrix Reloaded,Adventure Action Thriller Science Fiction,The Matrix Collection,0.400092
1,The Matrix Revolutions,Adventure Action Thriller Science Fiction,The Matrix Collection,0.389537
2,Battle of the Stars,Science Fiction Action,,0.385002
3,Road Wars,Science Fiction Action,,0.365762
4,The Phantom,Science Fiction Action,,0.365544
5,Earthfall,Action Science Fiction,,0.362926
6,Circuitry Man,Science Fiction Action,Circuitry Man Collection,0.362143
7,V: The Final Battle,Science Fiction,V: The Collection,0.362049
8,The Day the Sky Exploded,Science Fiction Action,,0.358390
9,Prince of Space,Action Science Fiction,,0.356508



REKOMENDASI UNTUK : Minions


,title,genres,belongs_to_collection,similarity_score
0,Despicable Me 2,Animation Comedy Family,Despicable Me Collection,0.571038
1,Despicable Me 3,Action Animation Adventure Family Comedy,Despicable Me Collection,0.443261
2,Despicable Me,Animation Family,Despicable Me Collection,0.439603
3,Banana,Animation Comedy Family,Despicable Me Collection,0.408550
4,Mower Minions,Comedy Animation,Despicable Me Collection,0.351468
5,Wreck-It Ralph,Family Animation Comedy Adventure,Wreck-It Ralph Collection,0.344241
6,The Curse of the Were-Rabbit,Adventure Animation Comedy Family,Wallace & Gromit Collection,0.342516
7,Pokémon: Giratina and the Sky Warrior,Adventure Animation Family,Pokémon Collection,0.341544
8,Dragon Ball Z: Broly - The Legendary Super Saiyan,Animation Adventure Family,Dragon Ball Z (Movie) Collection,0.340614
9,Dino Time,Animation Adventure Comedy Family,,0.340352



REKOMENDASI UNTUK : The Shawshank Redemption


,title,genres,belongs_to_collection,similarity_score
0,Brute Force,Crime Drama,,0.418596
1,Breath,Crime Drama,,0.398099
2,Duffy of San Quentin,Drama Crime,,0.393072
3,Brubaker,Crime Drama,,0.387561
4,Variety,Drama Crime,,0.386127
5,Cool Hand Luke,Crime Drama,,0.383035
6,Convict's Code,Crime Drama,,0.381893
7,A Prophet,Crime Drama,,0.381181
8,The Domino Principle,Drama Crime,,0.379429
9,House of Women,Crime Drama,,0.374816


In [71]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

movie_title = "The Dark Knight"

weights = {
    "genres": 0.30,
    "overview": 0.45,
    "tagline": 0.10,
    "belongs_to_collection": 0.15
}

matches = np.flatnonzero(
    content_df["title"].str.casefold().to_numpy()
    == movie_title.casefold()
)

if len(matches) == 0:
    print(f"Film '{movie_title}' tidak ditemukan.")
else:
    movie_idx = int(matches[0])
    contributions = {}

    for col, weight in weights.items():
        matrix = tfidf_matrices[col]
        similarity = cosine_similarity(
            matrix[movie_idx], matrix
        ).ravel()

        contributions[col] = weight * similarity

    diagnostic = content_df[
        ["title", "genres"]
    ].copy()

    for col, values in contributions.items():
        diagnostic[f"{col}_contribution"] = values

    contribution_cols = [
        f"{col}_contribution" for col in weights
    ]

    diagnostic["total_score"] = diagnostic[
        contribution_cols
    ].sum(axis=1)

    diagnostic = diagnostic.drop(index=content_df.index[movie_idx])

    display(
        diagnostic.nlargest(10, "total_score").reset_index(drop=True)
    )

,title,genres,genres_contribution,overview_contribution,tagline_contribution,belongs_to_collection_contribution,total_score
0,The Dark Knight Rises,Action Crime Drama Thriller,0.300000,0.126426,0.0,0.150000,0.576426
1,Batman Begins,Action Crime Drama,0.259591,0.080194,0.0,0.150000,0.489785
2,Ricochet,Action Crime Thriller,0.283673,0.078147,0.0,0.000000,0.361820
3,Mesrine: Killer Instinct,Drama Action Thriller Crime,0.300000,0.043736,0.0,0.012038,0.355774
4,Get the Gringo,Action Thriller Crime Drama,0.300000,0.044987,0.0,0.000000,0.344987
5,Running Out of Time,Drama Action Thriller Crime,0.300000,0.035308,0.0,0.009566,0.344874
6,Hitman,Action Crime Drama Thriller,0.300000,0.029515,0.0,0.012038,0.341553
7,S.W.A.T.,Action Thriller Crime,0.283673,0.019307,0.0,0.038517,0.341496
8,Mesrine: Public Enemy #1,Action Thriller Crime Drama,0.300000,0.026795,0.0,0.012038,0.338834
9,F/X,Crime Thriller Action,0.283673,0.014177,0.0,0.038517,0.336366


## Evaluasi Hasil Rekomendasi

### Tujuan Evaluasi

Evaluasi dilakukan untuk mengamati kemampuan sistem dalam merekomendasikan film berdasarkan kemiripan konten. Pengujian menggunakan beberapa film dari genre dan koleksi yang berbeda untuk melihat apakah rekomendasi yang dihasilkan relevan dengan film masukan.

### Hasil Pengamatan

Berdasarkan pengujian terhadap 10 film, sistem menunjukkan hasil yang cukup menjanjikan pada sejumlah contoh:

* **The Dark Knight:** merekomendasikan *The Dark Knight Rises* dan *Batman Begins* pada posisi teratas.
* **Finding Nemo:** merekomendasikan *Finding Dory* serta film animasi keluarga lainnya.
* **The Godfather:** merekomendasikan *The Godfather: Part II* dan *The Godfather: Part III*.
* **Toy Story:** merekomendasikan *Toy Story 3* dan *Toy Story 2*.
* **The Conjuring:** merekomendasikan *The Conjuring 2* dan sejumlah film horor.
* **The Matrix:** merekomendasikan dua film sekuelnya pada posisi teratas.
* **Minions:** merekomendasikan beberapa film dalam waralaba *Despicable Me*.
* **Home Alone:** merekomendasikan beberapa film dari koleksi yang sama.

Pada film *Titanic*, rekomendasi didominasi film bergenre drama dan romansa. Sementara itu, rekomendasi untuk *The Shawshank Redemption* mencakup beberapa film bergenre crime dan drama. Kesamaan genre tersebut belum tentu menunjukkan kesamaan alur cerita secara keseluruhan.

### Interpretasi

Hasil pengamatan menunjukkan bahwa kombinasi fitur genre, overview, tagline, dan koleksi film dapat menghasilkan rekomendasi yang memiliki kemiripan konten. Informasi koleksi film tampak membantu dalam menempatkan sekuel atau film dari waralaba yang sama pada posisi teratas.

Namun, evaluasi ini masih bersifat kualitatif. Pengamatan terhadap sejumlah contoh belum cukup untuk menyimpulkan bahwa sistem memiliki tingkat relevansi yang tinggi pada seluruh dataset.

### Keterbatasan

1. Sistem hanya memanfaatkan metadata dan konten film yang tersedia pada dataset.
2. Kualitas rekomendasi bergantung pada kelengkapan dan representasi teks setiap fitur.
3. Bobot fitur yang digunakan merupakan konfigurasi awal eksperimental dan belum dioptimalkan secara sistematis.
4. Evaluasi relevansi masih dilakukan secara kualitatif, belum menggunakan ground truth atau metrik evaluasi rekomendasi yang terukur.
5. Kemiripan berdasarkan teks tidak selalu mencerminkan kesamaan tema, alur cerita, atau preferensi pengguna.

### Kesimpulan

Content-Based Movie Recommender berhasil menghasilkan rekomendasi berdasarkan kemiripan metadata film dengan pendekatan TF-IDF dan cosine similarity. Hasil awal menunjukkan bahwa sistem dapat mengenali beberapa hubungan yang masuk akal, terutama berdasarkan genre dan koleksi film. Evaluasi lanjutan diperlukan untuk mengukur relevansi secara lebih objektif dan menentukan konfigurasi fitur yang lebih baik.
